In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## lite

In [ ]:
"""--------------------------------------------"""
# make sure that encoding weights make sense
# cvr2 and delta r2, oop-ify
# add movement (average normalized me on a trial)
"""--------------------------------------------"""
# add time
# one regressor

In [ ]:
from core.data import load_sess

# get data
(spike_times, trial_data, psths, session_data, regions) = load_sess(
    subj_id=subj_id,
    sess_id=sess_id,
    tpre=0.5,
    tpost=1,
    alignment="choice",
    tpre_ref=0.5,
    tpost_ref=1,
    alignment_ref="choice",
    binwidth_ms=25,
    thresh=1,
)

In [ ]:
from sg.fitlvm_utils import get_data_model
import numpy as np

# make robs
# when constructing the design matrix, add the idx as a drift term
(data_gd, train_dl, val_dl, test_dl, indices, num_trials, num_tv, num_units) = (
    get_data_model(
        psths,
        trial_data,
        strategy_filter=None,
        regions=regions,
        norm=True,
        num_tents=5,
        task_vars=[
            "response",
            "rewarded",
            "block_side",
            "response_prev",
            "rewarded_prev",
        ],
        sanity_check=0,
    )
)
sample = data_gd[:]
robs = sample["robs"].detach().cpu().numpy()
tvs = sample["tv"].detach().cpu().numpy()
tents = sample["tents"].detach().cpu().numpy()

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

n_folds = 10
n_samples = robs.shape[0]
p_train = 0.8

dm = np.hstack((tents, tvs))

scores_baseline_cv = np.zeros((n_folds, num_units))
scores_cv = np.zeros((n_folds, num_units))

for i in range(n_folds):
    train_idxs = np.sort(
        np.random.choice(n_samples, int(n_samples * p_train), replace=False)
    )
    test_idxs = np.setdiff1d(np.arange(n_samples), train_idxs)

    baseline_model = RidgeCV(
        alphas=np.logspace(-5, 5, 11, base=10),
        alpha_per_target=True,
    ).fit(tents[train_idxs], robs[train_idxs])

    scores_baseline_cv[i] = r2_score(
        robs[test_idxs],
        baseline_model.predict(tents[test_idxs]),
        multioutput="raw_values",
    )

    encoder = RidgeCV(
        alphas=np.logspace(-5, 5, 11, base=10),
        alpha_per_target=True,
    ).fit(dm[train_idxs], robs[train_idxs])

    scores_cv[i] = r2_score(
        robs[test_idxs], encoder.predict(dm[test_idxs]), multioutput="raw_values"
    )

scores_baseline = np.median(scores_baseline_cv, axis=0)
scores = np.median(scores_cv, axis=0)
robs_predict = encoder.predict(dm)

In [ ]:
# task responsive percentage (~50)
# replicate the baseline v task var plot

In [ ]:
plt.figure()
plt.scatter(scores_baseline, scores, alpha=0.5, s=0.5)
plt.plot([-0.2, 1], [-0.2, 1])
plt.show()

In [ ]:
np.round(
    np.where((scores > scores_baseline) & (scores > 0))[0].shape[0] / len(scores), 3
)

In [ ]:
np.where(scores > 0)[0].shape

In [ ]:
np.where(scores > scores_baseline)[0].shape

In [ ]:
np.where((scores > scores_baseline) & (scores > 0))[0].shape

In [ ]:
len(scores)

In [ ]:
98 / 131

In [ ]:
scores.mean()

In [ ]:
np.median(scores)